Cuando hacés requests.get(), lo que recibís en respuesta.text es un string plano (un texto larguísimo) que contiene todo el código HTML de la página. Para Python, eso es solo texto, no sabe qué es un título o un link.

BeautifulSoup(...): Es el constructor. Imaginalo como un traductor.

respuesta.text: Es el material bruto (el código HTML).

'html.parser': Es la herramienta que usa el traductor. Le dice a Python: "Leé esto siguiendo las reglas del lenguaje HTML".

Aquí estamos aplicando una técnica llamada encadenamiento. En lugar de buscar en toda la página, vamos cerrando el círculo:

.find('div', class_='side_categories'): Le decimos: "Buscá el primer contenedor <div> que tenga la etiqueta de clase side_categories". Ignoramos todo lo demás (banners, pies de página, etc.).

.find('ul'): Dentro de ese div, buscá la primera lista desordenada (<ul>). En este sitio, esa primera lista contiene el título "Books".

.find('ul'): Buscamos la segunda lista anidada. Aquí es donde están realmente las categorías (Travel, Mystery, etc.).

In [2]:
import requests
from bs4 import BeautifulSoup

# 1. Definimos la URL base
URL = "http://books.toscrape.com/"

def test_categorias():
    # 2. Hacemos la petición a la web
    print(f"Conectando a {URL}...")
    respuesta = requests.get(URL)
    
    # 3. Convertimos el HTML en algo que Python entienda (Soup)
    soup = BeautifulSoup(respuesta.text, 'html.parser')
    
    # 4. Buscamos el menú lateral de categorías
    # El sitio usa una lista <ul> dentro de un div con clase 'side_categories'
    menu_lateral = soup.find('div', class_='side_categories').find('ul').find('ul')
    categorias = menu_lateral.find_all('li')
    
    print(f"\n✅ ¡Conexión exitosa! Se encontraron {len(categorias)} categorías:")
    cant = 0
    # 5. Mostramos las categorias
    for cat in categorias:
        cant += 1
        nombre = cat.get_text(strip=True)
        link = cat.find('a')['href']
        print(f"{cant} - {nombre} (Ruta: {link})")

# Ejecutamos la prueba
if __name__ == "__main__":
    test_categorias()

Conectando a http://books.toscrape.com/...

✅ ¡Conexión exitosa! Se encontraron 50 categorías:
1 - Travel (Ruta: catalogue/category/books/travel_2/index.html)
2 - Mystery (Ruta: catalogue/category/books/mystery_3/index.html)
3 - Historical Fiction (Ruta: catalogue/category/books/historical-fiction_4/index.html)
4 - Sequential Art (Ruta: catalogue/category/books/sequential-art_5/index.html)
5 - Classics (Ruta: catalogue/category/books/classics_6/index.html)
6 - Philosophy (Ruta: catalogue/category/books/philosophy_7/index.html)
7 - Romance (Ruta: catalogue/category/books/romance_8/index.html)
8 - Womens Fiction (Ruta: catalogue/category/books/womens-fiction_9/index.html)
9 - Fiction (Ruta: catalogue/category/books/fiction_10/index.html)
10 - Childrens (Ruta: catalogue/category/books/childrens_11/index.html)
11 - Religion (Ruta: catalogue/category/books/religion_12/index.html)
12 - Nonfiction (Ruta: catalogue/category/books/nonfiction_13/index.html)
13 - Music (Ruta: catalogue/category/b

In [5]:
def extraer_detalle_libro(url_libro):
    r = requests.get(url_libro)
    # Forzamos la codificación correcta por si acaso
    r.encoding = 'utf-8' 
    soup = BeautifulSoup(r.text, 'html.parser')
    
    titulo = soup.find('h1').get_text(strip=True)
    precio_texto = soup.find('p', class_='price_color').get_text(strip=True)
    
    # NUEVA LIMPIEZA ROBUSTA
    precio_limpio = "".join(c for c in precio_texto if c.isdigit() or c == ".")
    precio = float(precio_limpio)

    tabla = soup.find('table', class_='table-striped')
    upc = ""
    for fila in tabla.find_all('tr'):
        encabezado = fila.find('th').get_text(strip=True)
        if encabezado == 'UPC':
            upc = fila.find('td').get_text(strip=True)
            break
            
    return {
        "titulo": titulo,
        "precio": precio,
        "upc": upc
    }

test_url = "http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(extraer_detalle_libro(test_url))

{'titulo': 'A Light in the Attic', 'precio': 51.77, 'upc': 'a897fe39b1053632'}


In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

# Configuración Global
BASE_URL = "http://books.toscrape.com/"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"}

def get_category_map():
    """Obtiene un diccionario con {Nombre de Categoría: URL completa}"""
    print("📋 Obteniendo categorías del sitio...")
    r = requests.get(BASE_URL, headers=HEADERS)
    soup = BeautifulSoup(r.text, 'html.parser')
    
    # Filtramos para encontrar el menú lateral
    # Buscamos el <ul> que está dentro del <ul> del div 'side_categories'
    categorias_html = soup.find('div', class_='side_categories').find('ul').find('ul').find_all('li')
    
    mapa_cats = {}
    for cat in categorias_html:
        nombre = cat.get_text(strip=True)
        link = urljoin(BASE_URL, cat.find('a')['href'])
        mapa_cats[nombre] = link
    return mapa_cats

def extraer_detalle_libro(url_libro):
    """Entra a la ficha de un libro y extrae Título, Precio y UPC"""
    r = requests.get(url_libro, headers=HEADERS)
    r.encoding = 'utf-8' # Evita errores de caracteres extraños como Â
    soup = BeautifulSoup(r.text, 'html.parser')
    
    # 1. Título: Etiqueta única h1
    titulo = soup.find('h1').get_text(strip=True)
    
    # 2. Precio: Buscamos clase específica y limpiamos el texto
    precio_raw = soup.find('p', class_='price_color').get_text(strip=True)
    # Solo conservamos números y puntos decimales (limpieza robusta)
    precio_limpio = "".join(c for c in precio_raw if c.isdigit() or c == ".")
    precio = float(precio_limpio)

    # 3. UPC: Buscamos en la tabla de información técnica
    tabla = soup.find('table', class_='table-striped')
    upc = "N/A"
    for fila in tabla.find_all('tr'):
        encabezado = fila.find('th').get_text(strip=True)
        if encabezado == 'UPC':
            upc = fila.find('td').get_text(strip=True)
            break
            
    return {
        "titulo": titulo,
        "precio": precio,
        "upc": upc
    }

def ejecutar_scraper_completo():
    """Función maestra que coordina todo el proceso"""
    start_time = time.time()
    
    # Paso 1: Mapeo
    categorias = get_category_map()
    base_datos_libros = []

    # Paso 2: Bucle por Categorías
    for nombre_cat, url_cat in categorias.items():
        print(f"\n--- 📂 Procesando: {nombre_cat} ---")
        
        url_actual = url_cat
        
        # Paso 3: Bucle de Paginación (Botón Next)
        while url_actual:
            r = requests.get(url_actual, headers=HEADERS)
            soup = BeautifulSoup(r.text, 'html.parser')
            
            # Buscamos todos los libros en la página actual
            articulos = soup.find_all('article', class_='product_pod')
            
            for art in articulos:
                # Obtenemos la URL del detalle del libro
                href = art.h3.a['href']
                url_detalle = urljoin(url_actual, href)
                
                try:
                    # Extraemos la información técnica
                    info_libro = extraer_detalle_libro(url_detalle)
                    
                    # Inyectamos la categoría que capturamos en el bucle superior
                    info_libro['categoria'] = nombre_cat
                    
                    # Guardamos el resultado
                    base_datos_libros.append(info_libro)
                    print(f"   ✔ {info_libro['titulo'][:40]}...")
                    
                    # Pausa mínima de cortesía al servidor
                    time.sleep(0.05) 
                    
                except Exception as e:
                    print(f"   ❌ Error en libro {url_detalle}: {e}")

            # Buscamos si existe el botón "Next" para seguir en esta categoría
            boton_next = soup.find('li', class_='next')
            if boton_next:
                url_actual = urljoin(url_actual, boton_next.a['href'])
            else:
                url_actual = None # Terminó la categoría

    end_time = time.time()
    duracion = (end_time - start_time) / 60
    print(f"\n✨ PROCESO FINALIZADO ✨")
    print(f"📚 Libros extraídos: {len(base_datos_libros)}")
    print(f"⏱ Tiempo total: {duracion:.2f} minutos")
    
    return base_datos_libros

# Ejecución principal
if __name__ == "__main__":
    resultado = ejecutar_scraper_completo()
    
    # Aquí podrías ver los primeros 5 resultados para verificar
    import json
    print("\nEjemplo de los primeros resultados:")
    print(json.dumps(resultado[:3], indent=4, ensure_ascii=False))

📋 Obteniendo categorías del sitio...

--- 📂 Procesando: Travel ---
   ✔ It's Only the Himalayas...
   ✔ Full Moon over Noah’s Ark: An Odyssey to...
   ✔ See America: A Celebration of Our Nation...
   ✔ Vagabonding: An Uncommon Guide to the Ar...
   ✔ Under the Tuscan Sun...
   ✔ A Summer In Europe...
   ✔ The Great Railway Bazaar...
   ✔ A Year in Provence (Provence #1)...
   ✔ The Road to Little Dribbling: Adventures...
   ✔ Neither Here nor There: Travels in Europ...
   ✔ 1,000 Places to See Before You Die...

--- 📂 Procesando: Mystery ---
   ✔ Sharp Objects...
   ✔ In a Dark, Dark Wood...
   ✔ The Past Never Ends...
   ✔ A Murder in Time...
   ✔ The Murder of Roger Ackroyd (Hercule Poi...
   ✔ The Last Mile (Amos Decker #2)...
   ✔ That Darkness (Gardiner and Renner #1)...
   ✔ Tastes Like Fear (DI Marnie Rome #3)...
   ✔ A Time of Torment (Charlie Parker #14)...
   ✔ A Study in Scarlet (Sherlock Holmes #1)...
   ✔ Poisonous (Max Revere Novels #3)...
   ✔ Murder at the 42nd Street L